# Market Index Prices

This notebook pre-processes the market index price dataset used to represent wholesale cost wihtin price optimisation

## Importing the Data and Dependencies

Importing required packages:

In [1]:
from google.colab import drive
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

Mounting to Google Drive:

In [2]:
drive.mount('/content/drive')

Mounted at /content/drive


Importing market index price data from https://bmrs.elexon.co.uk/market-index-prices for the 7-day period 24/07/25 - 31/07/25.

In [3]:
market_prices = pd.read_csv('/content/drive/MyDrive/ERP/Raw_Data/market_index_prices.csv')
market_prices.head()

,StartTime,DataProvider,SettlementDate,SettlementPeriod,Price,Volume
0,2025-07-31T00:00:00Z,APXMIDP,2025-07-31,3,70.36,1318.20
1,2025-07-31T00:00:00Z,N2EXMIDP,2025-07-31,3,0.00,0.00
2,2025-07-30T23:30:00Z,APXMIDP,2025-07-31,2,73.41,1626.35
3,2025-07-30T23:30:00Z,N2EXMIDP,2025-07-31,2,0.00,0.00
4,2025-07-30T23:00:00Z,APXMIDP,2025-07-31,1,77.45,1536.90


## Pre-processing

* Keeping observations from data provider APXMIDP only
* Irrelevant columns are removed
* Prices are converted from £/MWh to p/kWh

In [4]:
market_prices = market_prices[market_prices['DataProvider'] == 'APXMIDP']
market_prices = market_prices[['StartTime','Price']]
market_prices = market_prices.sort_values(by='StartTime')
market_prices['Price'] = market_prices['Price'] * 0.1
market_prices.head()

,StartTime,Price
672,2025-07-24T00:00:00Z,8.148
670,2025-07-24T00:30:00Z,7.933
668,2025-07-24T01:00:00Z,8.265
666,2025-07-24T01:30:00Z,7.878
664,2025-07-24T02:00:00Z,8.481


Averaging prices across hourly intervals to keep resolution consistent across datasets:

In [5]:
market_prices["StartTime"] = pd.to_datetime(market_prices["StartTime"], utc=True)
market_prices["Hour"] = market_prices["StartTime"].dt.floor("h")
hourly_prices = market_prices.groupby("Hour", as_index=False)["Price"].mean()

## Averaging Prices

Extracting the average prices as a numpy array:

In [6]:
hourly_prices["Hour"] = pd.to_datetime(hourly_prices["Hour"], utc=True)
hourly_prices["Day"] = hourly_prices["Hour"].dt.date
hourly_prices["Weekday"] = hourly_prices["Hour"].dt.weekday
weekdays = hourly_prices[hourly_prices["Weekday"] < 5].copy()
weekdays["x"] = weekdays["Hour"].dt.hour
average_prices = weekdays.groupby("x")["Price"].mean().to_numpy()


## Exporting the Data

In [7]:
pd.DataFrame(average_prices).to_csv('/content/drive/MyDrive/ERP/Processed_Data/wholesale_prices.csv')